# 07i — Train & Evaluate (Deep Graph CNN / SortPooling Readout Comparison, Pooled Multi-City)

Eighth parallel branch. Same pooled multi-city data, same plain random
70/15/15 split, same capacity numbers (`hidden_dim`, `fusion_dim`,
`head_hidden`, dropout, embedding dims), same 150-epoch budget
(`warmup_epochs=20`, `patience=30`), and `conv_type: gatv2` as `07g` --
**a clean, single-variable comparison**: only the **readout** differs.
`_pool_and_anchor` (concat of a global mean-pool with the anchor node's
own embedding) is swapped for `DGCNNReadout` (SortPooling, Zhang et al.
2018, "An End-to-End Deep Learning Architecture for Graph
Classification") on every encoder (`src/models.py`'s `readout` switch,
default `"pool_anchor"`, this run uses `"dgcnn"`).

**Reuses `configs/eval_capacity_revision.yaml` UNCHANGED** (not a copy --
same convention `07h`/GIN already established) -- so `07g` and `07i` use
IDENTICAL metrics and hyperparameters (150 epochs, same split, same
optimizer settings, same accuracy-default `primary_metric` inherited
from `train.py`, not config-driven here). `configs/model_dgcnn_comparison.yaml`
is `model_capacity_revision.yaml` plus exactly the keys `readout="dgcnn"`
needs (`svg_dgcnn_k`/`tvg_dgcnn_k`/`unified_dgcnn_k`, `conv2_kernel`
per encoder) -- readout is the only thing that can move the numbers here.

**How `k` is chosen -- not a guess.** SortPooling truncates/pads every
graph to a fixed `k` nodes (sorted descending by the last conv layer's
last channel). Zhang et al.'s own rule (Sec 4.2): choose `k` as the
**40th percentile** of the per-graph total node-count distribution, so
60% of graphs get truncated to their most salient `k` nodes and the
smaller 40% are fully retained (zero-padded). The diagnostic cell below
computes this **from real data**, per encoder, before any model is
built -- `configs/model_dgcnn_comparison.yaml`'s `svg_dgcnn_k`/
`tvg_dgcnn_k`/`unified_dgcnn_k` are placeholders only, overwritten here.

**Why TVG's second conv kernel (and therefore its `k` floor) differs
from SVG/Unified's.** The post-SortPool stack (`Conv1d` -> `MaxPool1d(2)`
-> `Conv1d(kernel=conv2_kernel)`) has a hard floor:
`k >= (conv2_kernel-1)*2 + 2`. The paper's own `conv2_kernel=5` implies
`k >= 10`. TVG's real graphs are small (this session's own QC data: mean
~8.1 total nodes across `incident`+`building`+`intersection`) --
*below* that floor. `tvg_dgcnn_conv2_kernel` is relaxed to `3`
(`k >= 6`) specifically for this reason; SVG and Unified keep the paper
default (`5`). Expect the diagnostic cell to report the FLOOR (not the
40th-percentile rule) as the deciding factor for TVG's `k` more often
than for SVG's/Unified's -- that's TVG's small graph scale showing up in
the numbers, not a bug.

**TVG-at-this-scale is a hypothesis under test, not a known-good
transplant.** DGCNN's original benchmarks (chemical/social graphs) run
dozens-to-hundreds of nodes per graph; at TVG's scale (~8 nodes), sort-
pooling mostly reduces to "sort a handful of real nodes, then pad the
rest" -- the mechanism's core value (picking the most salient `k` out of
a much larger pool) barely applies. A weak result on TVG/DGCNN here
should be read as "TVG graphs may be too small for sort-pooling to
help," not "DGCNN doesn't work."

**No anchor guarantee, unlike `07g`.** `_pool_and_anchor` always includes
the ego/incident node's own embedding in the readout. DGCNN's
sort-and-truncate has no such guarantee -- the anchor node can rank
outside the top-`k` and be dropped from the pooled sequence entirely. A
genuine architectural difference from `07g`, not an oversight.

**Per-point raw test predictions are recorded automatically** (same
mechanism as `07g` -- `train.py`'s `train_one_fold` writes a
`..._test_predictions.json` per scenario/repeat: `point_id`, `label`,
`prob`, `pred`, `category` = TP/TN/FP/FN). A cell near the end aggregates
all of them into one CSV, same as `07g`'s.

**Scenario G is skipped** -- XGBoost tabular baseline, no GNN encoder,
`readout` doesn't apply. `07g`'s `G` result is the one to use for any
G-inclusive comparison.

No formal significance testing here, same caveat as every random-repeats
branch: descriptive aggregates (mean +/- std across 5 repeats) only.

GPU recommended.

In [ ]:
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# TEMP: install locally patched src/ files until pushed to GitHub.
# Skip this cell once the repo itself is updated -- needs the revised
# models.py (DGCNNReadout + readout="dgcnn" switch) and train.py
# (accuracy-default primary_metric + per-point raw test predictions),
# plus graph_datasets.py, unified_graph.py, baseline_features.py,
# evaluate.py, plot_history.py.
from google.colab import files
import shutil

print("Upload train.py, models.py, graph_datasets.py, unified_graph.py, "
      "baseline_features.py, evaluate.py, plot_history.py:")
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, f"{REPO_DIR}/src/{fname}")
print("Patched files installed:", list(uploaded.keys()))

In [ ]:
!pip install -q torch_geometric xgboost scikit-learn scipy pyyaml pandas tqdm

In [ ]:
import yaml
from pathlib import Path
import torch

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)
# Reused UNCHANGED from 07g -- same eval scheme, only the model config
# differs. Same convention 07h (GIN) already established.
with open(f"{REPO_DIR}/configs/eval_capacity_revision.yaml") as f:
    eval_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/model_dgcnn_comparison.yaml") as f:
    model_cfg = yaml.safe_load(f)

# paths.yaml (current 4-city schema): interim_dir/processed_dir/outputs_dir
# are COMBINED (not per-city) roots at the top level -- every city's 01-04
# output already lands in ONE shared tree, disambiguated by the
# city-prefixed point_id/filename, not by directory. Same top-level keys
# 05/06/07g already read; no per-city base_dir, no "combined" sub-section.
CITIES = paths_cfg["cities"]
INTERIM_DIR = Path(paths_cfg["interim_dir"])
COMBINED_PROCESSED_DIR = Path(paths_cfg["processed_dir"])
OUTPUTS_DIR = Path(paths_cfg["outputs_dir"])
# separate checkpoint/metrics dirs from every other 07 branch, including 07g/07h
CHECKPOINT_DIR = OUTPUTS_DIR / "checkpoints_dgcnn_comparison"
METRICS_DIR = OUTPUTS_DIR / "metrics_dgcnn_comparison"
for d in [CHECKPOINT_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
# 07g's dirs, read-only here -- only used later by the comparison cell.
CAPACITY_REVISION_METRICS_DIR = OUTPUTS_DIR / "metrics_capacity_revision"

device = "cuda" if torch.cuda.is_available() else "cpu"
HEAD_DEPTH = model_cfg.get("head_depth", "mlp2")  # fixed, NOT swept -- see notebook intro
CONV_TYPE = model_cfg.get("conv_type", "gatv2")   # unchanged vs 07g -- readout is the intended variable
READOUT = model_cfg.get("readout", "dgcnn")
config = {"batch_size": eval_cfg.get("batch_size") or 128,
          "epoch_cap": eval_cfg.get("epoch_cap") or 150,
          "warmup_epochs": eval_cfg.get("warmup_epochs") or 20,
          "patience": eval_cfg.get("patience") or 30,
          "lr_patience": eval_cfg.get("lr_patience") or 8,
          "lr": eval_cfg.get("lr") or 5e-3,
          "weight_decay": eval_cfg.get("weight_decay") or 1e-4,
          "fusion_dim": model_cfg.get("fusion_dim") or 128,
          "head_hidden": model_cfg.get("head_hidden") or 32,
          "head_dropout": model_cfg.get("head_dropout") or 0.35,
          "val_frac": eval_cfg.get("val_frac") or 0.15,
          "test_frac": eval_cfg.get("test_frac") or 0.15,
          "label_col": eval_cfg.get("label_col") or "label",
          "target_pos_frac": eval_cfg.get("target_pos_frac"),
          "threshold_method": eval_cfg.get("threshold_method") or "fixed",
          "threshold_fn_cost": eval_cfg.get("threshold_fn_cost") or 10.0,
          "threshold_fp_cost": eval_cfg.get("threshold_fp_cost") or 1.0,
          "num_workers": eval_cfg.get("num_workers") or 0,
          "use_amp": eval_cfg.get("use_amp", True),
          # NOT set in eval_capacity_revision.yaml -- train.py's
          # train_one_fold defaults PRIMARY_METRIC to "accuracy" on its
          # own when this key is absent, same inherited default 07g uses.
          "primary_metric": eval_cfg.get("primary_metric", "accuracy")}
N_REPEATS = eval_cfg.get("n_repeats") or 5

train_frac = 1 - config["val_frac"] - config["test_frac"]
print("Device:", device, "| n_repeats:", N_REPEATS, "| head_depth:", HEAD_DEPTH, "(fixed, not swept)")
print("conv_type:", CONV_TYPE, "(unchanged vs 07g) | readout:", READOUT, "(the ONLY variable vs 07g)")
print("primary_metric:", config["primary_metric"], "(inherited default, same as 07g) | epoch_cap:",
      config["epoch_cap"], " warmup=", config["warmup_epochs"], " patience=", config["patience"], sep="")
print("Split:", f"{train_frac:.0%}/{config['val_frac']:.0%}/{config['test_frac']:.0%}",
      "(train/val/test), stratified by", config["label_col"], "ONLY -- plain random, NOT by city")
print("Capacity (identical to 07g): hidden_dim=", model_cfg.get("hidden_dim"), " fusion_dim=", config["fusion_dim"],
      " head_hidden=", config["head_hidden"], " dropout=", model_cfg.get("dropout"),
      " head_dropout=", config["head_dropout"], sep="")

In [ ]:
import json
import pandas as pd
import graph_datasets as ds
import train as tr
import evaluate as ev
import models
import unified_graph as ug

SVG_DIR = COMBINED_PROCESSED_DIR / "svg_graphs"
TVG_DIR = COMBINED_PROCESSED_DIR / "tvg_graphs"
INDEX_PATH = COMBINED_PROCESSED_DIR / "dataset_index.parquet"
index_df = pd.read_parquet(INDEX_PATH)
assert "city" in index_df.columns, (
    f"'{INDEX_PATH}' has no 'city' column -- this notebook needs 05's combined, "
    "multi-city dataset_index.parquet (all cities pooled into one index/one "
    "SVG_DIR/TVG_DIR, city-prefixed point_id), not a single-city index.")

# DualGraphDataset, not PooledDualGraphDataset -- the latter is vestigial
# from the old (deleted) 05b_dataset_assembly_pooled.ipynb design (per-row
# svg_dir/tvg_dir columns, a separate 'uid' column). Current 01-05 already
# gives every point_id a global city prefix (bog_/war_/kra_/som_) and
# writes every city's graphs into ONE shared SVG_DIR/TVG_DIR, so the plain
# single-directory dataset class (same one 05/06/07g already use) is
# correct here too -- point_id itself doubles as the globally-unique id.
dataset = ds.DualGraphDataset(index_df, SVG_DIR, TVG_DIR)
print(f"Dataset: {len(dataset)} points (pooled, DGCNN comparison branch)")
print(index_df.groupby("city")["label"].agg(["count", "sum"]).rename(columns={"sum": "n_positive"}))

# Unified vocab sizes, read from the post-04b cache -- same convention as
# 05/06/07g: any city's cache holds the identical unified vocab post-04b,
# so the first city in CITIES is as good as any other to read from.
# Confirm every OTHER city's cache agrees, catching a partially-run 04b
# before it silently misaligns the shared embedding table.
_ref_cache_dir = INTERIM_DIR / "osm_cache" / CITIES[0]
with open(_ref_cache_dir / "highway_vocab.json") as f:
    HIGHWAY_VOCAB_SIZE = len(json.load(f))
with open(_ref_cache_dir / "building_type_vocab.json") as f:
    BUILDING_TYPE_VOCAB_SIZE = len(json.load(f))
for city in CITIES[1:]:
    cache_dir = INTERIM_DIR / "osm_cache" / city
    with open(cache_dir / "highway_vocab.json") as f:
        hw_n = len(json.load(f))
    with open(cache_dir / "building_type_vocab.json") as f:
        bt_n = len(json.load(f))
    assert hw_n == HIGHWAY_VOCAB_SIZE and bt_n == BUILDING_TYPE_VOCAB_SIZE, (
        f"{city}'s vocab cache ({hw_n} highway / {bt_n} building_type) disagrees with "
        f"{CITIES[0]}'s ({HIGHWAY_VOCAB_SIZE} / {BUILDING_TYPE_VOCAB_SIZE}) -- run "
        f"04b_vocab_unification for every city before trusting this notebook.")
print(f"Unified vocab (post-04b, all {len(CITIES)} cities agree): "
      f"highway={HIGHWAY_VOCAB_SIZE}, building_type={BUILDING_TYPE_VOCAB_SIZE}")

## Diagnostic: compute real per-graph node counts, apply the k = max(floor, p40) rule

Run BEFORE building `svg_kwargs`/`tvg_kwargs` -- computes the actual
per-graph total node-count distribution (summed across every node type,
not per-type) for a sample of real pooled-dataset graphs, then applies
the rule from the notebook intro directly (not just prints percentiles
for manual eyeballing), overwriting `model_dgcnn_comparison.yaml`'s
placeholder `svg_dgcnn_k`/`tvg_dgcnn_k`/`unified_dgcnn_k` with the
data-computed values used for the rest of this run. Prints p40/p50/p75/
p90 plus which rule (architectural floor vs. 40th-percentile) determined
the final `k`, per encoder, so the choice is auditable from this cell's
own output.

In [ ]:
import numpy as np

def _total_node_count(hetero_data, node_types):
    return sum(int(hetero_data[nt].x.shape[0]) for nt in node_types if nt in hetero_data.node_types)

SAMPLE_N = min(500, len(dataset))
sample_idx = index_df.sample(n=SAMPLE_N, random_state=42).index

_tvg_node_types_no_peer = [nt for nt in models.TVG_NODE_TYPES if nt != "peer_incident"]

svg_counts, tvg_counts, unified_counts = [], [], []
for i in sample_idx:
    svg_d, tvg_d, _label, _pid = dataset[i]
    svg_counts.append(_total_node_count(svg_d, models.SVG_NODE_TYPES))
    tvg_counts.append(_total_node_count(tvg_d, _tvg_node_types_no_peer))
    merged_d = ug.merge_svg_tvg(svg_d, tvg_d)
    unified_counts.append(_total_node_count(merged_d, models.UNIFIED_NODE_TYPES))

def _compute_k(counts, conv2_kernel, label):
    counts = np.asarray(counts)
    p40, p50, p75, p90 = np.percentile(counts, [40, 50, 75, 90])
    floor_k = (conv2_kernel - 1) * 2 + 2  # DGCNNReadout's own __init__ floor assert
    k = max(floor_k, int(np.ceil(p40)))
    rule = "architectural floor" if floor_k >= p40 else "40th-percentile rule"
    print(f"{label:8s} | n={len(counts):4d} p40={p40:5.1f} p50={p50:5.1f} p75={p75:5.1f} p90={p90:5.1f} "
          f"| conv2_kernel={conv2_kernel} floor={floor_k:2d} -> k={k:3d} (determined by {rule})")
    return k

print(f"Computed from a random sample of {SAMPLE_N} pooled points (seed=42):\n")
SVG_DGCNN_K = _compute_k(svg_counts, model_cfg.get("svg_dgcnn_conv2_kernel", 5), "SVG")
TVG_DGCNN_K = _compute_k(tvg_counts, model_cfg.get("tvg_dgcnn_conv2_kernel", 3), "TVG")
UNIFIED_DGCNN_K = _compute_k(unified_counts, model_cfg.get("unified_dgcnn_conv2_kernel", 5), "Unified")

print(f"\nOverriding config placeholders: svg_dgcnn_k {model_cfg.get('svg_dgcnn_k')} -> {SVG_DGCNN_K}, "
      f"tvg_dgcnn_k {model_cfg.get('tvg_dgcnn_k')} -> {TVG_DGCNN_K}, "
      f"unified_dgcnn_k {model_cfg.get('unified_dgcnn_k')} -> {UNIFIED_DGCNN_K}")

In [ ]:
svg_kwargs = dict(hidden_dim=model_cfg.get("hidden_dim", 128), heads=model_cfg.get("heads", 4),
                   num_layers=model_cfg.get("svg_layers", 2), dropout=model_cfg.get("dropout", 0.3),
                   signage_vocab=5, light_pole_vocab=4, road_marking_vocab=2,
                   cat_embed_dim=model_cfg.get("cat_embed_dim", 4), conv_type=CONV_TYPE,
                   readout=READOUT, dgcnn_k=SVG_DGCNN_K,
                   dgcnn_conv2_kernel=model_cfg.get("svg_dgcnn_conv2_kernel", 5))
tvg_kwargs = dict(hidden_dim=model_cfg.get("hidden_dim", 128), heads=model_cfg.get("heads", 4),
                   num_layers=model_cfg.get("tvg_layers", 2), dropout=model_cfg.get("dropout", 0.3),
                   building_type_vocab=BUILDING_TYPE_VOCAB_SIZE, highway_vocab=HIGHWAY_VOCAB_SIZE,
                   building_type_embed_dim=model_cfg.get("building_type_embed_dim", 16),
                   highway_embed_dim=model_cfg.get("highway_embed_dim", 8), conv_type=CONV_TYPE,
                   readout=READOUT, dgcnn_k=TVG_DGCNN_K,
                   dgcnn_conv2_kernel=model_cfg.get("tvg_dgcnn_conv2_kernel", 3))
# UnifiedEncoder (scenario F) needs its OWN k, not whichever of svg/tvg's
# dgcnn_k would otherwise silently win build_model()'s kwarg merge --
# threaded through config so train_one_fold passes it to build_model
# every call (see build_model's own unified_dgcnn_k override).
config["unified_dgcnn_k"] = UNIFIED_DGCNN_K

print("svg_kwargs:", svg_kwargs)
print("tvg_kwargs:", tvg_kwargs)
print("unified_dgcnn_k:", config["unified_dgcnn_k"])

### Scenario A -- SVG only

In [ ]:
key = "A"
print(f"\n=== {key} ({HEAD_DEPTH}, {CONV_TYPE}, readout={READOUT}) ===")
results = tr.run_scenario_random_repeats("A", HEAD_DEPTH, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results = {key: results}
print(f"  {len(results)} repeat-runs complete.")

### Scenario B -- TVG only

In [ ]:
key = "B"
print(f"\n=== {key} ({HEAD_DEPTH}, {CONV_TYPE}, readout={READOUT}) ===")
results = tr.run_scenario_random_repeats("B", HEAD_DEPTH, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

### Scenario C -- dual graph (concat)

In [ ]:
key = "C"
print(f"\n=== {key} ({HEAD_DEPTH}, {CONV_TYPE}, readout={READOUT}) ===")
results = tr.run_scenario_random_repeats("C", HEAD_DEPTH, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

### Scenario D -- dual graph (late fusion)

In [ ]:
key = "D"
print(f"\n=== {key} ({HEAD_DEPTH}, {CONV_TYPE}, readout={READOUT}) ===")
results = tr.run_scenario_random_repeats("D", HEAD_DEPTH, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

### Scenario E -- dual graph (cross-attention)

In [ ]:
key = "E"
print(f"\n=== {key} ({HEAD_DEPTH}, {CONV_TYPE}, readout={READOUT}) ===")
results = tr.run_scenario_random_repeats("E", HEAD_DEPTH, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

### Scenario F -- unified merged graph

In [ ]:
key = "F"
print(f"\n=== {key} ({HEAD_DEPTH}, {CONV_TYPE}, readout={READOUT}) ===")
results = tr.run_scenario_random_repeats("F", HEAD_DEPTH, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

### Ablation B+ through F+

Same rationale as `07g`/`07h`: one cell per scenario, split out for
independent run/monitor/interrupt.

#### Ablation B+

In [ ]:
key = "B_ablation"
print(f"\n=== {key} ({HEAD_DEPTH}, {CONV_TYPE}, readout={READOUT}) ===")
results = tr.run_scenario_random_repeats("B", HEAD_DEPTH, use_ablation=True, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

#### Ablation C+

In [ ]:
key = "C_ablation"
print(f"\n=== {key} ({HEAD_DEPTH}, {CONV_TYPE}, readout={READOUT}) ===")
results = tr.run_scenario_random_repeats("C", HEAD_DEPTH, use_ablation=True, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

#### Ablation D+

In [ ]:
key = "D_ablation"
print(f"\n=== {key} ({HEAD_DEPTH}, {CONV_TYPE}, readout={READOUT}) ===")
results = tr.run_scenario_random_repeats("D", HEAD_DEPTH, use_ablation=True, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

#### Ablation E+

In [ ]:
key = "E_ablation"
print(f"\n=== {key} ({HEAD_DEPTH}, {CONV_TYPE}, readout={READOUT}) ===")
results = tr.run_scenario_random_repeats("E", HEAD_DEPTH, use_ablation=True, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

#### Ablation F+

In [ ]:
key = "F_ablation"
print(f"\n=== {key} ({HEAD_DEPTH}, {CONV_TYPE}, readout={READOUT}) ===")
results = tr.run_scenario_random_repeats("F", HEAD_DEPTH, use_ablation=True, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

### Scenario G -- skipped on this branch

G is the XGBoost tabular baseline (`baseline_features.py`), no GNN
encoder involved at all -- `readout` has nothing to act on. `07g`'s `G`
result is the one to use for any comparison that needs to include it;
not re-run here.

## Aggregate + report every scenario

In [ ]:
agg_rows = []
for key, results in all_results.items():
    agg = ev.aggregate_fold_results(results)
    row = {"scenario": key}
    for metric, (mean, std) in agg.items():
        row[f"{metric}_mean"] = mean
        row[f"{metric}_std"] = std
    agg_rows.append(row)

agg_df = pd.DataFrame(agg_rows)
agg_df.to_csv(METRICS_DIR / "all_scenarios_summary_dgcnn_comparison.csv", index=False)
display(agg_df)

## Threshold diagnostics

In [ ]:
threshold_rows = []
for key, results in all_results.items():
    method_counts = ev.summarize_categorical_field(results, "threshold_method")
    thresh_mean, thresh_std = ev.aggregate_fold_results(results).get("threshold_used", (float("nan"), float("nan")))
    threshold_rows.append({"scenario": key, "threshold_mean": thresh_mean, "threshold_std": thresh_std,
                            "methods_used": method_counts})

threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(METRICS_DIR / "threshold_diagnostics_dgcnn_comparison.csv", index=False)
display(threshold_df)

## Epoch-level diagnostics

Per-repeat training history saved under
`CHECKPOINT_DIR/{tag}_history/repeat{N}.json`, same JSON shape as every
other branch -- including `val_accuracy` alongside `val_pr_auc`/
`val_auroc`.

In [ ]:
import json
import matplotlib.pyplot as plt

history_path = CHECKPOINT_DIR / f"A_{HEAD_DEPTH}_history" / "repeat0.json"
history = json.loads(history_path.read_text())

epochs = [h["epoch"] for h in history]
best_epoch = max(range(len(history)), key=lambda i: history[i]["val_pr_auc"])

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(epochs, [h["train_loss"] for h in history], label="train_loss", color="tab:blue")
ax1.set_xlabel("epoch"); ax1.set_ylabel("train_loss", color="tab:blue")

ax2 = ax1.twinx()
ax2.plot(epochs, [h.get("val_accuracy") for h in history], label="val_accuracy (selection metric)", color="tab:red")
ax2.plot(epochs, [h["val_pr_auc"] for h in history], label="val_pr_auc", color="tab:orange")
ax2.plot(epochs, [h["val_auroc"] for h in history], label="val_auroc", color="tab:green")
ax2.axvline(best_epoch, color="gray", linestyle="--", label=f"best epoch ({best_epoch})")
ax2.set_ylabel("val metric")

fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.9))
plt.title(f"repeat0 history ({history_path.name}, readout={READOUT}, primary_metric={config['primary_metric']})")
plt.tight_layout()
plt.show()

## Per-point raw predictions (TP/TN/FP/FN), aggregated across every scenario/repeat

Same mechanism as `07g` -- walks every
`{tag}_history/repeat{N}_test_predictions.json` under `CHECKPOINT_DIR`,
tags each row with its `scenario`/`repeat`, and concatenates everything
into ONE CSV, so specific points can be compared directly against
`07g`'s own `raw_test_predictions_all_scenarios.csv` (e.g. "which points
does GATv2+pool_anchor get right that DGCNN gets wrong, or vice versa").

In [ ]:
import re

raw_pred_rows = []
for history_dir in CHECKPOINT_DIR.glob("*_history"):
    tag = history_dir.name[: -len("_history")]
    for pred_file in history_dir.glob("repeat*_test_predictions.json"):
        m = re.match(r"repeat(\d+)_test_predictions\.json", pred_file.name)
        repeat_idx = int(m.group(1)) if m else -1
        records = json.loads(pred_file.read_text())
        for r in records:
            raw_pred_rows.append({"scenario": tag, "repeat": repeat_idx, **r})

if raw_pred_rows:
    raw_pred_df = pd.DataFrame(raw_pred_rows)
    raw_pred_df.to_csv(METRICS_DIR / "raw_test_predictions_all_scenarios.csv", index=False)
    print(f"Aggregated {len(raw_pred_df)} per-point predictions across "
          f"{raw_pred_df['scenario'].nunique()} scenario/ablation tags, "
          f"{raw_pred_df['repeat'].nunique()} repeats each.")
    display(raw_pred_df.groupby("scenario")["category"].value_counts().unstack(fill_value=0))
else:
    print("No test_predictions.json files found yet -- run scenario cells above first.")

## Validation + test metrics side by side, aggregated across every scenario/repeat

`train_one_fold` (and scenario G's own loop above) each write a
`..._val_test_metrics.json` per (scenario, repeat): the FULL metric
suite (PR-AUC, AUROC, accuracy, F1, precision, recall, BCE loss,
confusion counts, threshold used) AND per-point raw predictions, for
BOTH the validation split and the test split, at the exact same
(best-epoch) weights -- not just the single scalar tracked during
training. This cell walks every `{tag}_history/repeat{N}_val_test_metrics.json`
under `CHECKPOINT_DIR` and concatenates every split's metrics into ONE
CSV, tagged by scenario/repeat/split, so validation and test performance
can be compared directly rather than validation only ever appearing as
an implicit selection signal.

In [ ]:
val_test_rows = []
for history_dir in CHECKPOINT_DIR.glob("*_history"):
    tag = history_dir.name[: -len("_history")]
    for vt_file in history_dir.glob("repeat*_val_test_metrics.json"):
        m = re.match(r"repeat(\d+)_val_test_metrics\.json", vt_file.name)
        repeat_idx = int(m.group(1)) if m else -1
        payload = json.loads(vt_file.read_text())
        for split in ("val", "test"):
            val_test_rows.append({"scenario": tag, "repeat": repeat_idx, "split": split,
                                   **payload[split]["metrics"]})

if val_test_rows:
    val_test_df = pd.DataFrame(val_test_rows)
    val_test_df.to_csv(METRICS_DIR / "val_test_metrics_all_scenarios.csv", index=False)
    print(f"Aggregated {len(val_test_df)} val+test metric rows across "
          f"{val_test_df['scenario'].nunique()} scenario/ablation tags.")
    display(val_test_df.groupby(["scenario", "split"])[["accuracy", "pr_auc", "auroc"]].mean())
else:
    print("No val_test_metrics.json files found yet -- run scenario cells above first.")

## DGCNN (SortPooling) vs GATv2+pool_anchor -- direct comparison against 07g

Loads `07g`'s own summary CSV (same capacity numbers, same 150-epoch
split, same hyperparameters, GATv2+pool_anchor) alongside this run's,
merged on `scenario`. Since `07i` and `07g` share every hyperparameter
except readout, this is a genuine single-variable comparison -- any
delta below is attributable to the readout swap alone.

In [ ]:
capacity_revision_summary_path = CAPACITY_REVISION_METRICS_DIR / "all_scenarios_summary_capacity_revision.csv"
if capacity_revision_summary_path.exists():
    baseline_df = pd.read_csv(capacity_revision_summary_path)
    compare_cols = ["scenario", "pr_auc_mean", "pr_auc_std", "auroc_mean", "auroc_std",
                     "accuracy_mean", "accuracy_std"]
    compare_cols = [c for c in compare_cols if c in agg_df.columns and c in baseline_df.columns or c == "scenario"]
    compare = agg_df[compare_cols].merge(baseline_df[compare_cols], on="scenario", suffixes=("_dgcnn", "_gatv2_pool"))
    if "pr_auc_mean_dgcnn" in compare.columns:
        compare["pr_auc_delta_dgcnn_minus_gatv2pool"] = compare["pr_auc_mean_dgcnn"] - compare["pr_auc_mean_gatv2_pool"]
    if "accuracy_mean_dgcnn" in compare.columns:
        compare["accuracy_delta_dgcnn_minus_gatv2pool"] = compare["accuracy_mean_dgcnn"] - compare["accuracy_mean_gatv2_pool"]
    compare.to_csv(METRICS_DIR / "dgcnn_vs_gatv2pool_comparison.csv", index=False)
    display(compare)
    if "accuracy_delta_dgcnn_minus_gatv2pool" in compare.columns:
        n_dgcnn_better = (compare["accuracy_delta_dgcnn_minus_gatv2pool"] > 0).sum()
        print(f"DGCNN readout has higher mean accuracy than GATv2+pool_anchor on {n_dgcnn_better}/{len(compare)} scenarios.")
else:
    print("07g's summary CSV not found yet -- run 07g first to compare.")

In [ ]:
print("DGCNN (SortPooling) comparison branch complete.")
print(f"Every scenario A-F (+ ablations) trained with readout={READOUT}, conv_type={CONV_TYPE},")
print(f"primary_metric={config['primary_metric']}, epoch_cap={config['epoch_cap']} -- identical")
print("hyperparameters to 07g except readout.")
print("k values used (data-computed, see the diagnostic cell above): "
      f"svg={SVG_DGCNN_K} tvg={TVG_DGCNN_K} unified={UNIFIED_DGCNN_K}")
print("See the comparison cell above (or metrics_dgcnn_comparison/dgcnn_vs_gatv2pool_comparison.csv)")
print("for whether the readout swap actually moved accuracy/PR-AUC/AUROC, and")
print("raw_test_predictions_all_scenarios.csv for point-by-point comparisons.")